# MOSAIC-ML — Stage 2: ML Model Comparison on MH-100K**Purpose.** Train and fairly compare several classifiers on the MH-100K Androidmalware dataset, and report the accuracy (and much more) of each.**This notebook reports measured results only.** Nothing is pre-filled. Everynumber in the final table comes from a model trained in this runtime.### Honesty controls built into this notebookThree design choices exist to stop the comparison flattering itself:1. **Deduplication.** 42.54% of MH-100K rows are duplicates. Splitting at random   puts copies of the same app in both train and test, which inflates every   score. This notebook splits on *unique* feature rows by default.2. **A count-only control model.** Malware in MH-100K has a median of 42 active   features; benign has 2,082. A model using only *how many* features are active   — ignoring which — scores about 95% accuracy. It is included so you can see   how much the 24,833 features actually add.3. **A sealed test split.** Models are tuned and selected on validation data.   The test split is evaluated exactly once, at the end.### Label definitionLabels are `mh100.npz['classes']`, which is exactly `vt_detection >= 4`(verified at 100.00% agreement). The `CLASS` column in `mh100_labels.csv`disagrees on 10,505 rows and matches no threshold — it is **not** used.> Runtime: use **Runtime → Change runtime type → High-RAM** if available.> The dense matrix is 2.53 GB; this notebook converts it to sparse in chunks.

## 1. Setup

In [ ]:
import sys, os, time, json, hashlib, platformimport numpy as npimport scipy.sparse as spimport matplotlibimport matplotlib.pyplot as pltimport sklearnprint("python      :", sys.version.split()[0])print("numpy       :", np.__version__)print("scipy       :", sp.__version__ if hasattr(sp,'__version__') else __import__('scipy').__version__)print("scikit-learn:", sklearn.__version__)print("platform    :", platform.platform())SEED = 42np.random.seed(SEED)plt.rcParams.update({'font.size': 10, 'figure.dpi': 130})

## 2. Load the datasetPut `mh100.npz` in your Google Drive, then set `NPZ_PATH` below.The file is ~86 MB.

In [ ]:
USE_DRIVE = Trueif USE_DRIVE:    from google.colab import drive    drive.mount('/content/drive')    NPZ_PATH = '/content/drive/MyDrive/MOSAIC-ML/mh100.npz'else:    from google.colab import files    up = files.upload()    NPZ_PATH = list(up.keys())[0]assert os.path.exists(NPZ_PATH), f"Not found: {NPZ_PATH}"print("Using:", NPZ_PATH, f"({os.path.getsize(NPZ_PATH)/1e6:.1f} MB)")

In [ ]:
t0 = time.time()z = np.load(NPZ_PATH, allow_pickle=True)print("arrays:", list(z.keys()))y_all   = np.asarray(z['classes']).astype(int)vocab   = np.asarray(z['column_names'])sha_all = np.asarray(z['sha256'])dense = z['data']print("dense matrix:", dense.shape, dense.dtype, f"({dense.nbytes/1e9:.2f} GB)")# Chunked conversion avoids a second full-size allocation.blocks, CH = [], 5000for s in range(0, dense.shape[0], CH):    blocks.append(sp.csr_matrix(dense[s:s+CH]))X_all = sp.vstack(blocks, format='csr')del blocks, denseimport gc; gc.collect()print(f"CSR: {X_all.shape}, nnz={X_all.nnz:,}, density={X_all.nnz/(X_all.shape[0]*X_all.shape[1]):.4%}")print(f"loaded in {time.time()-t0:.1f}s")

### Verify the published structure (166 permissions + 250 intents + 24,417 API calls)

In [ ]:
from collections import Counterblocks_count = Counter(str(v).split('::')[0] for v in vocab)print("vocabulary blocks:", dict(blocks_count))print("total features   :", len(vocab))expected = {'Permission': 166, 'Intent': 250, 'APICall': 24417}print("matches published:", {k: blocks_count.get(k) == v for k, v in expected.items()})n0, n1 = int((y_all==0).sum()), int((y_all==1).sum())print(f"\nbenign  (vt<4) : {n0:,} ({100*n0/len(y_all):.2f}%)")print(f"malware (vt>=4): {n1:,} ({100*n1/len(y_all):.2f}%)")print(f"imbalance ratio : {n0/n1:.2f} : 1")

## 3. DeduplicateRows are duplicates when they have an identical set of active feature indices.Set `DEDUPLICATE = False` to reproduce the original (inflated) protocol andcompare the two directly.

In [ ]:
DEDUPLICATE = Truedef row_digest(M, i):    lo, hi = M.indptr[i], M.indptr[i+1]    return hashlib.blake2b(M.indices[lo:hi].astype(np.int32).tobytes(), digest_size=16).digest()seen, keep, conflicts = {}, [], 0for i in range(X_all.shape[0]):    d = row_digest(X_all, i)    if d in seen:        if y_all[seen[d]] != y_all[i]:            conflicts += 1    else:        seen[d] = i        keep.append(i)keep = np.array(keep)print(f"total rows      : {X_all.shape[0]:,}")print(f"unique rows     : {len(keep):,}")print(f"duplicate rows  : {X_all.shape[0]-len(keep):,} ({100*(X_all.shape[0]-len(keep))/X_all.shape[0]:.2f}%)")print(f"duplicate pairs with CONFLICTING labels: {conflicts:,}")if DEDUPLICATE:    X, y, sha = X_all[keep], y_all[keep], sha_all[keep]    print(f"\n--> using {X.shape[0]:,} UNIQUE rows")else:    X, y, sha = X_all, y_all, sha_all    print(f"\n--> using all {X.shape[0]:,} rows (duplicates retained; scores will be optimistic)")print(f"class balance after: benign {int((y==0).sum()):,} / malware {int((y==1).sum()):,}")

## 4. Split: train / validation / testStratified 70 / 15 / 15 with a fixed seed. **The test split is not toucheduntil the final section.** All tuning and all threshold choices use validation only.

In [ ]:
from sklearn.model_selection import train_test_splitidx = np.arange(X.shape[0])idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, random_state=SEED, stratify=y)idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.50, random_state=SEED, stratify=y[idx_tmp])X_tr, y_tr = X[idx_tr], y[idx_tr]X_va, y_va = X[idx_va], y[idx_va]X_te, y_te = X[idx_te], y[idx_te]for nm, yy in [('train', y_tr), ('val', y_va), ('test', y_te)]:    print(f"{nm:6s}: {len(yy):6,}  benign {int((yy==0).sum()):6,}  malware {int((yy==1).sum()):5,}  ({100*(yy==1).mean():.2f}% malware)")SPLIT_INFO = {'seed': SEED, 'scheme': '70/15/15 stratified', 'deduplicated': DEDUPLICATE,              'n_train': len(y_tr), 'n_val': len(y_va), 'n_test': len(y_te)}

## 5. Define the modelsAll models receive the **same** sparse input and the **same** splits.`LinearSVC` has no `predict_proba`, so its `decision_function` is used forranking metrics — it is not treated as a calibrated probability.

In [ ]:
from sklearn.ensemble import RandomForestClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.svm import LinearSVCfrom sklearn.naive_bayes import BernoulliNBfrom sklearn.neighbors import KNeighborsClassifierMODELS = {    'Random Forest':        RandomForestClassifier(n_estimators=100, max_features='sqrt',                                                   random_state=SEED, n_jobs=-1),    'Random Forest (30)':   RandomForestClassifier(n_estimators=30, max_features='sqrt',                                                   random_state=SEED, n_jobs=-1),    'Logistic Regression':  LogisticRegression(max_iter=2000, random_state=SEED),    'Decision Tree':        DecisionTreeClassifier(random_state=SEED),    'Linear SVM':           LinearSVC(random_state=SEED, max_iter=3000),    'Bernoulli NB':         BernoulliNB(),}# KNN on 24,833 sparse dimensions is expensive. Off by default.INCLUDE_KNN = Falseif INCLUDE_KNN:    MODELS['KNN (k=5)'] = KNeighborsClassifier(n_neighbors=5, metric='jaccard', n_jobs=-1)print(f"{len(MODELS)} models queued:")for k in MODELS: print("  -", k)

### The control modelThis one ignores *which* features are active and uses only *how many*.If it scores close to the full models, the 24,833 features are adding littleand the dataset contains a shortcut. Treat it as the floor any real modelmust clear.

In [ ]:
def active_counts(M):    return np.diff(M.indptr).astype(float).reshape(-1, 1)cnt_tr, cnt_va, cnt_te = active_counts(X_tr), active_counts(X_va), active_counts(X_te)print("median active features per app")print(f"  benign : {np.median(cnt_tr[y_tr==0]):.0f}")print(f"  malware: {np.median(cnt_tr[y_tr==1]):.0f}")

## 6. Train and evaluate on validationFit times and inference times are measured here.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,                             precision_score, recall_score, confusion_matrix,                             roc_auc_score, average_precision_score)def scores_from(model, Xs):    """Continuous malware score, or None when the model offers neither."""    if hasattr(model, 'predict_proba'):        return model.predict_proba(Xs)[:, 1]    if hasattr(model, 'decision_function'):        return model.decision_function(Xs)    return Nonedef evaluate(name, model, Xa, ya, Xb, yb, note=''):    t0 = time.time(); model.fit(Xa, ya); fit_s = time.time() - t0    t1 = time.time(); pred = model.predict(Xb); inf_s = time.time() - t1    sc = scores_from(model, Xb)    tn, fp, fn, tp = confusion_matrix(yb, pred, labels=[0,1]).ravel()    row = {        'model': name,        'accuracy':          accuracy_score(yb, pred),        'balanced_accuracy': balanced_accuracy_score(yb, pred),        'macro_f1':          f1_score(yb, pred, average='macro'),        'weighted_f1':       f1_score(yb, pred, average='weighted'),        'malware_precision': precision_score(yb, pred, zero_division=0),        'malware_recall':    recall_score(yb, pred, zero_division=0),        'malware_f1':        f1_score(yb, pred, zero_division=0),        'false_positive_rate': fp / (fp + tn) if (fp+tn) else np.nan,        'false_negative_rate': fn / (fn + tp) if (fn+tp) else np.nan,        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),        'roc_auc': roc_auc_score(yb, sc) if sc is not None else np.nan,        'pr_auc':  average_precision_score(yb, sc) if sc is not None else np.nan,        'fit_seconds': round(fit_s, 2), 'inference_seconds': round(inf_s, 3),        'note': note,    }    return row, model, pred, scval_rows, fitted, val_scores = [], {}, {}for name, mdl in MODELS.items():    print(f"training {name} ...", end=' ', flush=True)    try:        row, m, pred, sc = evaluate(name, mdl, X_tr, y_tr, X_va, y_va)        val_rows.append(row); fitted[name] = m; val_scores[name] = sc        print(f"acc={row['accuracy']:.4f}  bal_acc={row['balanced_accuracy']:.4f}  "              f"mal_F1={row['malware_f1']:.4f}  ({row['fit_seconds']}s)")    except Exception as e:        print(f"FAILED: {type(e).__name__}: {e}")        val_rows.append({'model': name, 'note': f'FAILED: {type(e).__name__}: {e}'})# controlprint("training Active-count control ...", end=' ', flush=True)row, m, pred, sc = evaluate('CONTROL: active-count only',                            DecisionTreeClassifier(max_depth=3, random_state=SEED),                            cnt_tr, y_tr, cnt_va, y_va,                            note='1 feature: number of active features. Shortcut baseline.')val_rows.append(row); fitted['CONTROL: active-count only'] = m; val_scores['CONTROL: active-count only'] = scprint(f"acc={row['accuracy']:.4f}  bal_acc={row['balanced_accuracy']:.4f}  mal_F1={row['malware_f1']:.4f}")

### Validation comparison table

In [ ]:
import pandas as pdpd.set_option('display.width', 200, 'display.max_columns', 40)val_df = pd.DataFrame(val_rows).set_index('model')show = ['accuracy','balanced_accuracy','macro_f1','malware_precision','malware_recall',        'malware_f1','false_positive_rate','false_negative_rate','roc_auc','pr_auc','fit_seconds']display(val_df[show].round(4).sort_values('balanced_accuracy', ascending=False))print("\nAccuracy alone is misleading at "      f"{100*(y_tr==1).mean():.1f}% malware: predicting 'benign' for everything "      f"already scores {100*(y_va==0).mean():.2f}%.")

### How much do the 24,833 features actually add?Compare the best model against the one-number control.

In [ ]:
ctrl = val_df.loc['CONTROL: active-count only']real = val_df.drop(index='CONTROL: active-count only')best = real['balanced_accuracy'].idxmax()print(f"best model : {best}")for metric in ['accuracy','balanced_accuracy','malware_f1','malware_recall']:    b, c = real.loc[best, metric], ctrl[metric]    print(f"  {metric:18s}: best {b:.4f}   control {c:.4f}   gain {b-c:+.4f}")print("\nA small gain means the dataset is separable mostly by how FEW features "      "a malware sample has, which reflects failed API-call extraction rather "      "than learned malicious behaviour. Report this in threats to validity.")

## 7. Select the model on validation, then evaluate ONCE on testSelection uses balanced accuracy, not accuracy.

In [ ]:
SELECTED = bestprint("selected on validation:", SELECTED)final_model = MODELS[SELECTED] if SELECTED in MODELS else Noneassert final_model is not Nonet0 = time.time(); final_model.fit(X_tr, y_tr); fit_s = time.time()-t0pred_te = final_model.predict(X_te)sc_te = scores_from(final_model, X_te)tn, fp, fn, tp = confusion_matrix(y_te, pred_te, labels=[0,1]).ravel()test_row = {    'model': SELECTED, 'split': 'TEST (evaluated once)',    'accuracy': accuracy_score(y_te, pred_te),    'balanced_accuracy': balanced_accuracy_score(y_te, pred_te),    'macro_f1': f1_score(y_te, pred_te, average='macro'),    'malware_precision': precision_score(y_te, pred_te, zero_division=0),    'malware_recall': recall_score(y_te, pred_te, zero_division=0),    'malware_f1': f1_score(y_te, pred_te, zero_division=0),    'false_positive_rate': fp/(fp+tn), 'false_negative_rate': fn/(fn+tp),    'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),    'roc_auc': roc_auc_score(y_te, sc_te) if sc_te is not None else np.nan,    'pr_auc': average_precision_score(y_te, sc_te) if sc_te is not None else np.nan,    'fit_seconds': round(fit_s,2),}for k, v in test_row.items():    print(f"  {k:20s}: {v}")print(f"\nMalware missed (false negatives): {fn} of {fn+tp}")print(f"Benign wrongly flagged (false positives): {fp} of {fp+tn}")

## 8. Figures

In [ ]:
# Figure 4 - model comparisonplot_df = val_df.drop(columns=['note'], errors='ignore').dropna(subset=['accuracy'])metrics = ['balanced_accuracy','macro_f1','malware_recall','false_positive_rate']titles  = ['Balanced accuracy','Macro F1','Malware recall','False positive rate']fig, axes = plt.subplots(2, 2, figsize=(12, 7))order = plot_df['balanced_accuracy'].sort_values().indexfor ax, met, ttl in zip(axes.ravel(), metrics, titles):    vals = plot_df.loc[order, met]    cols = ['#C44E52' if 'CONTROL' in str(i) else '#4C72B0' for i in order]    ax.barh(range(len(vals)), vals.values, color=cols)    ax.set_yticks(range(len(vals))); ax.set_yticklabels(order, fontsize=8)    ax.set_title(ttl, fontsize=10); ax.grid(axis='x', alpha=.3)    for i, v in enumerate(vals.values):        if not np.isnan(v): ax.text(v, i, f' {v:.3f}', va='center', fontsize=7)fig.suptitle('Figure 4 - Model comparison on the VALIDATION split (red = shortcut control)', fontsize=11)fig.tight_layout(); plt.show()

In [ ]:
# Figure 5 - confusion matricesvalid = [n for n in fitted if n in val_df.index and not pd.isna(val_df.loc[n,'accuracy'])]ncol = 3; nrow = int(np.ceil(len(valid)/ncol))fig, axes = plt.subplots(nrow, ncol, figsize=(4*ncol, 3.4*nrow))for ax, name in zip(np.ravel(axes), valid):    r = val_df.loc[name]    cm = np.array([[r['TN'], r['FP']],[r['FN'], r['TP']]], dtype=int)    ax.imshow(cm, cmap='Blues')    for (i,j), v in np.ndenumerate(cm):        ax.text(j, i, f'{v:,}', ha='center', va='center',                color='white' if v > cm.max()/2 else 'black', fontsize=9)    ax.set_xticks([0,1]); ax.set_xticklabels(['pred benign','pred malware'], fontsize=8)    ax.set_yticks([0,1]); ax.set_yticklabels(['true benign','true malware'], fontsize=8)    ax.set_title(name, fontsize=9)for ax in np.ravel(axes)[len(valid):]: ax.axis('off')fig.suptitle('Figure 5 - Confusion matrices (validation split, raw counts)', fontsize=11)fig.tight_layout(); plt.show()

In [ ]:
# Figure 6 - ROC and precision-recallfrom sklearn.metrics import roc_curve, precision_recall_curvefig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6))for name, sc in val_scores.items():    if sc is None: continue    fpr, tpr, _ = roc_curve(y_va, sc)    a1.plot(fpr, tpr, lw=1.3, ls='--' if 'CONTROL' in name else '-',            label=f"{name} (AUC {roc_auc_score(y_va, sc):.3f})")    pr, rc, _ = precision_recall_curve(y_va, sc)    a2.plot(rc, pr, lw=1.3, ls='--' if 'CONTROL' in name else '-',            label=f"{name} (AP {average_precision_score(y_va, sc):.3f})")a1.plot([0,1],[0,1],'k:',lw=1,label='no skill')a1.set_xlabel('False positive rate'); a1.set_ylabel('True positive rate'); a1.set_title('ROC')a1.legend(fontsize=7); a1.grid(alpha=.3)a2.axhline((y_va==1).mean(), color='k', ls=':', lw=1, label=f'no skill ({(y_va==1).mean():.3f})')a2.set_xlabel('Recall'); a2.set_ylabel('Precision'); a2.set_title('Precision-Recall (better for imbalance)')a2.legend(fontsize=7); a2.grid(alpha=.3)fig.suptitle('Figure 6 - ROC and PR curves (validation split)', fontsize=11)fig.tight_layout(); plt.show()

## 9. Save resultsWrites the comparison table and a run record so every figure and number can beregenerated and traced.

In [ ]:
OUT = '/content/drive/MyDrive/MOSAIC-ML/results' if USE_DRIVE else '/content/results'os.makedirs(OUT, exist_ok=True)val_df.to_csv(f'{OUT}/model_comparison_validation.csv')pd.DataFrame([test_row]).to_csv(f'{OUT}/final_test_result.csv', index=False)record = {    'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),    'split': SPLIT_INFO,    'label_rule': "mh100.npz['classes'] == (vt_detection >= 4)",    'deduplicated': DEDUPLICATE,    'selected_model': SELECTED,    'selection_metric': 'balanced_accuracy on validation',    'test_evaluated_once': True,    'versions': {'python': sys.version.split()[0], 'numpy': np.__version__,                 'sklearn': sklearn.__version__},    'n_features': int(X.shape[1]),    'validation_results': val_df.reset_index().to_dict(orient='records'),    'test_result': test_row,    'caveats': [        'Single split. No repeated runs or confidence intervals.',        'Probabilities are uncalibrated; no calibration curve was produced.',        'MH-100K covers 2010-2022; generalisation to current APKs is untested.',        '10,992 apps flagged by 1-3 VT engines are labelled benign (greyware).',        '172 apps with vt_detection = -1 are labelled benign.',        'Compare every model against the active-count control before claiming the features matter.',    ],}with open(f'{OUT}/stage2_run_record.json','w') as f:    json.dump(record, f, indent=2, default=str)print("saved to", OUT)for f_ in os.listdir(OUT): print("  -", f_)

## 9b. Save the trained model for deploymentThe comparison above measures models but saves none of them. This cell trainsthe selected model on **train + validation** (the test split stays untouched;it has already been used once and must not inform the model) and writes adeployment bundle.**Why this matters.** A model saved by one scikit-learn version and loaded byanother raises `InconsistentVersionWarning` and is not guaranteed to behaveidentically. Training here, in the environment you will deploy from, removesthat mismatch. The bundle records the version so the warning can never be asurprise later.

In [ ]:
import joblib, sklearn, json, os, time# Refit on train + validation. The test split is NOT used - it was already# spent on the single final evaluation.import scipy.sparse as spX_fit = sp.vstack([X_tr, X_va], format='csr')y_fit = np.concatenate([y_tr, y_va])print(f"refitting {SELECTED} on {X_fit.shape[0]:,} rows (train + validation)")deploy_model = MODELS[SELECTED]t0 = time.time()deploy_model.fit(X_fit, y_fit)print(f"done in {time.time()-t0:.1f}s")os.makedirs(f'{OUT}/model', exist_ok=True)model_path = f'{OUT}/model/mosaic_model.joblib'joblib.dump(deploy_model, model_path, compress=3)size_mb = os.path.getsize(model_path) / 1e6print(f"saved {model_path} ({size_mb:.1f} MB)")metadata = {    'model_type': type(deploy_model).__name__,    'selected_as': SELECTED,    'selection_metric': 'balanced_accuracy on validation',    'sklearn_version': sklearn.__version__,    'numpy_version': np.__version__,    'n_features': int(X_fit.shape[1]),    'n_training_rows': int(X_fit.shape[0]),    'fitted_on': 'train + validation (test split untouched)',    'label_rule': 'vt_detection >= 4',    'deduplicated': DEDUPLICATE,    'split_seed': SEED,    'test_metrics': test_row,    'warning': (        'Load this model with scikit-learn ' + sklearn.__version__ +        '. A different version may change behaviour.'    ),}with open(f'{OUT}/model/model_metadata.json', 'w') as f:    json.dump(metadata, f, indent=2, default=str)print("\nmetadata:")for k, v in metadata.items():    if k != 'test_metrics':        print(f"  {k}: {v}")

### Verify the saved model loads and predictsNever ship a model you have not loaded back. A file that saves without errorcan still fail to load.

In [ ]:
reloaded = joblib.load(model_path)probe = sp.csr_matrix((1, X_fit.shape[1]), dtype=np.int8)probe[0, :50] = 1print("reloaded:", type(reloaded).__name__)print("expects features:", reloaded.n_features_in_)print("classes:", reloaded.classes_)print("probe prediction:", reloaded.predict(probe))print("probe probabilities:", reloaded.predict_proba(probe))assert reloaded.n_features_in_ == 24833, "feature count must match the vocabulary"print("\nOK - this model is safe to copy to ml/random_forest_model.joblib")print("Download it from the Files panel, or find it in your Drive at:")print(" ", model_path)

## 10. Reading these results**For the accuracy question.** The validation table in section 6 gives theaccuracy of every model. Quote **balanced accuracy and malware recall alongsideaccuracy** — at ~9.6% malware, predicting "benign" for everything already scoresabout 90%, so raw accuracy flatters every model.**The number that matters most** is the gap between the best model and theactive-count control in section 6. That gap is the real contribution of the24,833 features.**What these numbers do not establish.** They are measured on MH-100K(2010–2022) under one split. They say nothing about how the deployed MOSAIC-MLscanner behaves on an uploaded APK, because the feature mapper currentlyreaches only the 166 permission features — 99.33% of the vocabulary is notpopulated at scan time. Accuracy measured here is an upper bound on datasetperformance, not a claim about the live system.